# FlashAttention forward and backward in Triton

This Colab accompanies **Scaling Long-Context Attention, Part I**. We will run the same teaching implementation used by the article and check four claims:

1. tiled online softmax matches dense PyTorch attention;
2. the saved LSE matches `torch.logsumexp`;
3. backward reconstructs each probability tile instead of loading a stored \(L\times L\) matrix;
4. the Triton gradients match PyTorch autograd for both dense and causal attention.

The kernels are educational rather than production-tuned. They support contiguous `[batch, heads, sequence, head_dim]` fp16/bf16 tensors with head dimension 64 or 128.

## 0. Select a GPU runtime

In Colab, choose **Runtime → Change runtime type → GPU** before running the next cell. Triton requires an NVIDIA GPU.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("triton") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "triton"])

import torch
import triton

assert torch.cuda.is_available(), "Select a GPU runtime, then rerun this cell."
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("Triton:", triton.__version__)

## 1. Load the complete kernels

The complete forward, backward, autograd wrapper, and PyTorch checks live in one adjacent Python file. Keeping that file as the single source of truth prevents the article, standalone script, and Colab from silently diverging. The downloaded file also appears in Colab's Files pane, where we can open and edit every kernel.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import importlib

SOURCE_URL = (
    "https://raw.githubusercontent.com/G-U-N/G-U-N.github.io/"
    "master/blogs/code/flash_attention_backward.py"
)
SOURCE_PATH = Path("flash_attention_backward.py")
urlretrieve(SOURCE_URL, SOURCE_PATH)

fa = importlib.import_module("flash_attention_backward")
print(f"Loaded {SOURCE_PATH} ({SOURCE_PATH.stat().st_size:,} bytes)")

## 2. Forward reconstructs dense attention from tiles

The Triton kernel never materializes the complete score or probability matrix in HBM. It streams K/V tiles while each query tile keeps only its running row maximum, exponential sum, and output numerator on chip. We first compare its output and saved LSE with a dense PyTorch reference.

In [ ]:
torch.manual_seed(0)
B, H, L, D = 1, 2, 128, 64
dtype = torch.float16

q = torch.randn((B, H, L, D), device="cuda", dtype=dtype) * 0.5
k = torch.randn_like(q) * 0.5
v = torch.randn_like(q) * 0.5

o_tri, lse_tri = fa.flash_forward(q, k, v, causal=False)
o_ref, scores_ref = fa._reference(q, k, v, causal=False)
lse_ref = torch.logsumexp(scores_ref, dim=-1)

print("max |O_triton - O_torch|:", (o_tri - o_ref).abs().max().item())
print("max |LSE_triton - LSE_torch|:", (lse_tri - lse_ref).abs().max().item())
torch.testing.assert_close(o_tri, o_ref, atol=3e-2, rtol=3e-2)
torch.testing.assert_close(lse_tri, lse_ref, atol=3e-2, rtol=3e-2)
print("Forward and LSE checks passed.")

## 3. LSE is the compact bridge to backward

For one row, `P = exp(S - LSE)`. Saving one LSE scalar per query row lets backward reconstruct the current probability tile after recomputing its score tile. The full \(S\) and \(P\) matrices are never saved.

In [ ]:
example_L = 4096
full_s_and_p = 2 * B * H * example_L * example_L
saved_lse = B * H * example_L
print(f"Full S and P: {full_s_and_p:,} values")
print(f"Saved LSE:    {saved_lse:,} values")
print(f"Ratio:        {full_s_and_p / saved_lse:,.0f}x")

## 4. Backward matches PyTorch autograd

The backward kernel first forms \(D_i=dO_i^\top O_i\). It then makes one KV-owned traversal for complete `dK`/`dV` tiles and one query-owned traversal for complete `dQ` tiles. Both traversals recompute scores and recover probabilities from LSE.

In [ ]:
do = torch.randn_like(q)

q_tri, k_tri, v_tri = [x.detach().clone().requires_grad_(True) for x in (q, k, v)]
out_tri = fa.flash_attention(q_tri, k_tri, v_tri, False)
grads_tri = torch.autograd.grad(out_tri, (q_tri, k_tri, v_tri), do)

q_ref, k_ref, v_ref = [x.detach().clone().requires_grad_(True) for x in (q, k, v)]
out_ref, _ = fa._reference(q_ref, k_ref, v_ref, causal=False)
grads_ref = torch.autograd.grad(out_ref, (q_ref, k_ref, v_ref), do)

for name, actual, expected in zip(("dQ", "dK", "dV"), grads_tri, grads_ref):
    error = (actual - expected).abs().max().item()
    print(f"max |{name}_triton - {name}_torch|: {error}")
    torch.testing.assert_close(actual, expected, atol=5e-2, rtol=5e-2)
print("Backward checks passed.")

## 5. Run the complete dense and causal test matrix

The final check repeats forward, LSE, and backward comparisons for dense and causal attention. It also checks bf16 when the selected GPU supports it.

In [ ]:
for causal in (False, True):
    fa.check_against_torch(causal=causal, dtype=torch.float16)
    print(f"fp16 causal={causal}: passed")

if torch.cuda.is_bf16_supported():
    for causal in (False, True):
        fa.check_against_torch(causal=causal, dtype=torch.bfloat16)
        print(f"bf16 causal={causal}: passed")
else:
    print("This GPU does not support bf16; bf16 checks skipped.")

print("All available checks passed.")

## What to carry forward

The important result is not merely that a custom kernel matches PyTorch. Forward and backward both preserve the same dense attention function while changing which tiles live on chip and which intermediate values reach HBM. Forward saves only `O` and one LSE scalar per row; backward spends extra matrix multiplications to reconstruct the probability tiles it needs.